In [1]:
import psycopg2
import os
from dotenv import load_dotenv
import csv
from datetime import datetime, timedelta

# Load environment variables from .env
load_dotenv()

# Database connection information
DB_USERNAME = os.getenv("DB_USERNAME")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

def write_data_to_csv(table_name, output_file):
    try:
        # Establish a connection to the PostgreSQL database
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            database=DB_NAME,
            user=DB_USERNAME,
            password=DB_PASSWORD
        )

        # Create a cursor object
        cursor = conn.cursor()

        # Query the data from the specified table
        cursor.execute(f"SELECT start_date, end_date, interval, data FROM {table_name}")

        # Fetch all rows
        rows = cursor.fetchall()

        if rows:
            # Prepare data for CSV
            csv_data = []
            for row in rows:
                start_date, end_date, interval, data = row
                current_date = start_date
                interval_seconds = int(interval.total_seconds())

                for day_data in data:
                    current_date += timedelta(seconds=interval_seconds)

                    # Replace None with 0.0
                    day_data = [0.0 if value is None else value for value in day_data]

                    # Append each weather data value along with the timestamp
                    for value in day_data:
                        csv_data.append([current_date, value])

            # Write data to CSV file
            with open(output_file, "w", newline="") as csv_file:
                csv_writer = csv.writer(csv_file)
                csv_writer.writerow(["Timestamp", "Weather Data"])  # CSV header
                csv_writer.writerows(csv_data)

            print(f"CSV file '{output_file}' created successfully.")
        else:
            print(f"No data found in table '{table_name}'.")

        # Close the cursor and the connection
        cursor.close()
        conn.close()

    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    # Specify the table name and output file name
    table_name = "total_precipitation_aladin"  # Replace with your table name
    output_file = "./data/output.csv"  # Replace with your desired output file name

    write_data_to_csv(table_name, output_file)


CSV file './data/output.csv' created successfully.
